# Multi Marginal Optimal Transport

In [1]:
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
from dataset_OT import make_multi_WSI_dataset

seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 



In [2]:
#data loading


batch_size = 64 

print('Loading starting...')
idx_range_subset1 = [i for i in range(1,52+1)]
random.shuffle(idx_range_subset1)
num_train = int(np.ceil(0.7 * len(idx_range_subset1))) 
train_range1, val_range1 = idx_range_subset1[:num_train], idx_range_subset1[num_train:]

idx_range_subset3 = [i for i in range(1,26+1)] 
random.shuffle(idx_range_subset3)
num_train = int(np.ceil(0.7 * len(idx_range_subset3))) 
train_range3, val_range3 = idx_range_subset3[:num_train], idx_range_subset3[num_train:]

akoya_loader_train_subset1 = make_multi_WSI_dataset('Subset1', train_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset1 = make_multi_WSI_dataset('Subset1', val_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_train_subset3 = make_multi_WSI_dataset('Subset3', train_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset3 = make_multi_WSI_dataset('Subset3', val_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
leica_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
leica_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)

akoya_loader_train = ConcatDataset([akoya_loader_train_subset1, akoya_loader_train_subset3])
akoya_loader_val = ConcatDataset([akoya_loader_val_subset1, akoya_loader_val_subset3])

len_akoya_train = len(akoya_loader_train)
len_akoya_val = len(akoya_loader_val)

len_leica_train = len(leica_loader_train)
len_leica_val = len(leica_loader_val)

len_kfbio_train = len(kfbio_loader_train)
len_kfbio_val = len(kfbio_loader_val)

len_train = len_akoya_train + len_leica_train + len_kfbio_train
len_val = len_akoya_val + len_leica_val + len_kfbio_val

B_A_train = round(batch_size * len_akoya_train / (len_akoya_train + len_leica_train))
B_L_train = batch_size - B_A_train 
B_A_val = round(batch_size * len_akoya_val / (len_akoya_val + len_leica_val))
B_L_val = batch_size - B_A_val

akoya_loader_train = DataLoader(akoya_loader_train, batch_size=B_A_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
akoya_loader_val = DataLoader(akoya_loader_val, batch_size=B_A_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_train = DataLoader(leica_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_val = DataLoader(leica_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_train = DataLoader(kfbio_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_val = DataLoader(kfbio_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)

print("Train batches Akoya:", len(akoya_loader_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_loader_train), 'batch size:', B_L_train)
print("Train batches KFBio:", len(kfbio_loader_train), 'batch size:', B_L_train)
print("Validation batches Akoya:", len(akoya_loader_val), 'batch size:', B_A_val)
print("Validation batches Leica:", len(leica_loader_val), 'batch size:', B_L_val)

#in samples
print('len train:', len_train)



######
batch_size = 512 


def data_train_val(subset, ids, scanner):
    random.shuffle(ids)
    num_train = int(np.ceil(0.7 * len(ids))) 
    train_range, val_range = ids[:num_train], ids[num_train:]
    #print(f'for {scanner}, train range is {train_range}, val range is {val_range}')
    train_dataset = make_multi_WSI_dataset(subset, train_range, [scanner], train_or_test='Train', batch_size=batch_size)
    val_dataset = make_multi_WSI_dataset(subset, val_range, [scanner], train_or_test='Train', batch_size=batch_size)
    return train_dataset, val_dataset

akoya_data_train_subset1, akoya_data_val_subset1 = data_train_val('Subset1', [i for i in range(1,52+1)], 'Akoya')
akoya_data_train_subset3, akoya_data_val_subset3 = data_train_val('Subset3', [i for i in range(1,26+1)], 'Akoya')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
philips_data_train, philips_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Philips')
olympus_data_train, olympus_data_val = data_train_val('Subset3', [i for i in range(1,26+1) if i != 20], 'Olympus')
zeiss_data_train, zeiss_data_val = data_train_val('Subset3', [1,5,6,7,8,9,10,11,12,13,14,16,21,23,25], 'Zeiss')

akoya_data_train = ConcatDataset([akoya_data_train_subset1, akoya_data_train_subset3])
akoya_data_val = ConcatDataset([akoya_data_val_subset1, akoya_data_val_subset3])

len_akoya_train = len(akoya_data_train)
len_akoya_val = len(akoya_data_val)

len_leica_train = len(leica_data_train)
len_leica_val = len(leica_data_val)

len_philips_train = len(philips_data_train) 
len_philips_val = len(philips_data_val)

len_olympus_train = len(olympus_data_train)
len_olympus_val = len(olympus_data_val)

len_zeiss_train = len(zeiss_data_train)
len_zeiss_val = len(zeiss_data_val)

len_train = len_akoya_train + len_leica_train + len_philips_train + len_olympus_train + len_zeiss_train
len_val = len_akoya_val + len_leica_val + len_philips_val + len_olympus_val + len_zeiss_val

#batch sizes for train
B_A_train = round(batch_size * len_akoya_train / len_train)
B_L_train = round(batch_size * len_leica_train / len_train)
B_P_train = round(batch_size * len_philips_train / len_train)
B_O_train = round(batch_size * len_olympus_train / len_train)
B_Z_train = batch_size - B_A_train - B_L_train - B_P_train - B_O_train


#batch sizes for validation
B_A_val = round(batch_size * len_akoya_val / len_val)
B_L_val = round(batch_size * len_leica_val / len_val)
B_P_val = round(batch_size * len_philips_val / len_val)
B_O_val = round(batch_size * len_olympus_val / len_val)
B_Z_val = batch_size - B_A_val - B_L_val - B_P_val - B_O_val


def make_loader(dataset, auto_batch_size):
    return DataLoader(dataset, batch_size=auto_batch_size, shuffle=True, num_workers=1, pin_memory=True, persistent_workers=True, prefetch_factor=4)

akoya_loader_train = make_loader(akoya_data_train, B_A_train)
leica_loader_train = make_loader(leica_data_train, B_L_train)
philips_loader_train = make_loader(philips_data_train, B_P_train)
olympus_loader_train = make_loader(olympus_data_train, B_O_train)
zeiss_loader_train = make_loader(zeiss_data_train, B_Z_train)



print("Train batches Akoya:", len(akoya_data_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_data_train), 'batch size:', B_L_train)
print("Train batches Philips:", len(philips_data_train), 'batch size:', B_P_train)
print("Train batches olympus:", len(olympus_data_train), 'batch size:', B_O_train)
print("Train batches Zeiss:", len(zeiss_data_train), 'batch size:', B_Z_train)
print('len train:', len_train)


akoya_loader_val = make_loader(akoya_data_val, B_A_val)
leica_loader_val = make_loader(leica_data_val, B_L_val)
philips_loader_val = make_loader(philips_data_val, B_P_val)
olympus_loader_val = make_loader(olympus_data_val, B_O_val)
zeiss_loader_val = make_loader(zeiss_data_val, B_Z_val)

print("Val batches Akoya:", len(akoya_data_val), 'batch size:', B_A_val)
print("Val batches Leica:", len(leica_data_val), 'batch size:', B_L_val)
print("Val batches Philips:", len(philips_data_val), 'batch size:', B_P_val)
print("Val batches olympus:", len(olympus_data_val), 'batch size:', B_O_val)
print("Val batches Zeiss:", len(zeiss_data_val), 'batch size:', B_Z_val)
print('len train:', len_val)

Loading starting...
Train batches Akoya: 39681 batch size: 50
Train batches Leica: 40173 batch size: 14
Train batches KFBio: 48926 batch size: 14
Validation batches Akoya: 10072 batch size: 49
Validation batches Leica: 10467 batch size: 15
len train: 3231403
Train batches Akoya: 1991781 batch size: 250
Train batches Leica: 486822 batch size: 61
Train batches Philips: 621274 batch size: 78
Train batches olympus: 472204 batch size: 59
Train batches Zeiss: 501423 batch size: 64
len train: 4073504
Val batches Akoya: 485739 batch size: 198
Val batches Leica: 232590 batch size: 95
Val batches Philips: 198252 batch size: 81
Val batches olympus: 191969 batch size: 78
Val batches Zeiss: 144684 batch size: 60
len train: 1253234


In [3]:
import torch

def pairwise_sqdist(x, y):
    # x: (n,d), y: (m,d)
    x2 = (x**2).sum(dim=1).unsqueeze(1)  # (n,1)
    y2 = (y**2).sum(dim=1).unsqueeze(0)  # (1,m)
    xy = x @ y.T                      # (n,m)
    return x2 + y2 - 2*xy             # (n,m)

def make_broadcast_shape(sizes, axis):
    shape = [1] * len(sizes)
    shape[axis] = sizes[axis]
    return tuple(shape)

def multimarginal_sinkhorn_torch(as_list, C, eps=0.1, n_iters=50, atol=1e-12, renormalize_u=True):
    """
    Multimarginal Sinkhorn in PyTorch.

    as_list: list of 1D torch tensors (probability marginals, sum=1)
    C: cost tensor (shape matches sizes of as_list)
    eps: entropic regularization parameter
    n_iters: number of iterations
    """
    S = len(as_list)
    sizes = [a.shape[0] for a in as_list]

    # Kernel
    K = torch.exp(-C / eps)

    # Scaling vectors
    u_list = [torch.ones(n, device=C.device, dtype=C.dtype) for n in sizes]
    b_shapes = [make_broadcast_shape(sizes, s) for s in range(S)]

    for _ in range(n_iters):
        for s in range(S):
            Q = K.clone()
            for r in range(S):
                if r == s: 
                    continue
                Q *= u_list[r].reshape(b_shapes[r])

            axes_to_sum = tuple(i for i in range(S) if i != s)
            m_s = Q.sum(dim=axes_to_sum)
            m_s = torch.clamp(m_s, min=atol)

            u_new = as_list[s] / m_s
            if renormalize_u:
                u_new = u_new / (u_new.sum() + atol)
            u_list[s] = u_new

    P = K.clone()
    for s in range(S):
        P *= u_list[s].reshape(b_shapes[s])

    P = P / (P.sum() + atol)
    ot_cost = torch.sum(P * C)
    return ot_cost, u_list


In [4]:
# Test

sup_times = []




tq = tqdm(zip(akoya_loader_train, leica_loader_train, philips_loader_train, olympus_loader_train, zeiss_loader_train),
                                                    desc=f"Timing OT losses")
                                                    #total=min(len(akoya_loader_train), len(leica_loader_train), len(kfbio_loader_train)))

for batch_akoya, batch_leica, batch_philips, batch_olympus, batch_zeiss in tq:
    print('emb')
    emb_akoya = batch_akoya['embedding']   
    emb_leica = batch_leica['embedding']   
    emb_philips = batch_philips['embedding'] 
    emb_olympus = batch_olympus['embedding']
    emb_zeiss = batch_zeiss['embedding']

    # uniform marginals
    t0 = time.time()
    a = torch.full((emb_akoya.shape[0],), 1/emb_akoya.shape[0], device=emb_akoya.device)
    b = torch.full((emb_leica.shape[0],), 1/emb_leica.shape[0], device=emb_leica.device)
    c = torch.full((emb_philips.shape[0],), 1/emb_philips.shape[0], device=emb_philips.device)
    d = torch.full((emb_olympus.shape[0],), 1/emb_olympus.shape[0], device=emb_olympus.device)
    e = torch.full((emb_zeiss.shape[0],), 1/emb_zeiss.shape[0], device=emb_zeiss.device)

    print('tensor')

    # cost tensor
    C_al = pairwise_sqdist(emb_akoya, emb_leica)
    C_ap = pairwise_sqdist(emb_akoya, emb_philips)
    C_ao = pairwise_sqdist(emb_akoya, emb_olympus)
    C_az = pairwise_sqdist(emb_akoya, emb_zeiss)
    
    C_lp = pairwise_sqdist(emb_leica, emb_philips)
    C_lo = pairwise_sqdist(emb_leica, emb_olympus)
    C_lz = pairwise_sqdist(emb_leica, emb_zeiss)
    
    C_po = pairwise_sqdist(emb_philips, emb_olympus)
    C_pz = pairwise_sqdist(emb_philips, emb_zeiss)
    
    C_oz = pairwise_sqdist(emb_olympus, emb_zeiss)
    
    #C = C_xy.unsqueeze(2) + C_xz.unsqueeze(1) + C_yz.unsqueeze(0)
    print(C.shape)    
    C = C / C.max()

    #print("C min:", C.min().item(), "C max:", C.max().item())
    ot_loss, _ = multimarginal_sinkhorn_torch([a, b, c, d, e], C, eps=0.1, n_iters=1)
    t1 = time.time()

    sup_times.append(t1 - t0)
    avg_time = sum(sup_times) / len(sup_times)
    tq.set_description(f"Timing OT losses | Avg time: {avg_time:.3f}s | MMOT: {ot_loss.item():.4f}")
    
        

Timing OT losses: 0it [00:25, ?it/s]


KeyboardInterrupt: 

In [ ]:
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
import os
from typing import (
    Tuple, 
    Literal
)

from dataset_OT import make_multi_WSI_dataset
from architecture_OT import Network
from handler_OT import train_plot, end_epoch, better_confusion_matrix, dim_reduc_plot

seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 


batch_size = 512 

ids_subset1 = [i for i in range(1,52+1)]
random.shuffle(ids_subset1)
num_train = int(np.ceil(0.7 * len(ids_subset1))) 
train_range1, val_range1 = ids_subset1[:num_train], ids_subset1[num_train:]

ids_subset3 = [i for i in range(1,26+1)]
random.shuffle(ids_subset3)
num_train = int(np.ceil(0.7 * len(ids_subset3))) 
train_range3, val_range3 = ids_subset3[:num_train], ids_subset3[num_train:]


def data_train_val(subset, ids, scanner):
    if subset == 'Subset1':
        train_range, val_range = train_range1, val_range1
    else:
        train_range = [i for i in train_range3 if i in ids]
        val_range = [i for i in val_range3 if i in ids]

    #train_range = [1]
    #val_range = [5]
    
    train_dataset = make_multi_WSI_dataset(subset, train_range, [scanner], train_or_test='Train', batch_size=batch_size)
    val_dataset = make_multi_WSI_dataset(subset, val_range, [scanner], train_or_test='Train', batch_size=batch_size)
    return train_dataset, val_dataset

akoya_data_train_subset1, akoya_data_val_subset1 = data_train_val('Subset1', [i for i in range(1,52+1)], 'Akoya')
akoya_data_train_subset3, akoya_data_val_subset3 = data_train_val('Subset3', [i for i in range(1,26+1)], 'Akoya')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
leica_data_train, leica_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Leica')
philips_data_train, philips_data_val = data_train_val('Subset3', [i for i in range(1,26+1)], 'Philips')
olympus_data_train, olympus_data_val = data_train_val('Subset3', [i for i in range(1,26+1) if i != 20], 'Olympus')
zeiss_data_train, zeiss_data_val = data_train_val('Subset3', [1,5,6,7,8,9,10,11,12,13,14,16,21,23,25], 'Zeiss')

akoya_data_train = ConcatDataset([akoya_data_train_subset1, akoya_data_train_subset3])
akoya_data_val = ConcatDataset([akoya_data_val_subset1, akoya_data_val_subset3])

len_akoya_train = len(akoya_data_train)
len_akoya_val = len(akoya_data_val)

len_leica_train = len(leica_data_train)
len_leica_val = len(leica_data_val)

len_philips_train = len(philips_data_train) 
len_philips_val = len(philips_data_val)

len_olympus_train = len(olympus_data_train)
len_olympus_val = len(olympus_data_val)

len_zeiss_train = len(zeiss_data_train)
len_zeiss_val = len(zeiss_data_val)

len_train = len_akoya_train + len_leica_train + len_philips_train + len_olympus_train + len_zeiss_train
len_val = len_akoya_val + len_leica_val + len_philips_val + len_olympus_val + len_zeiss_val

#batch sizes for train
B_A_train = round(batch_size * len_akoya_train / len_train)
B_L_train = round(batch_size * len_leica_train / len_train)
B_P_train = round(batch_size * len_philips_train / len_train)
B_O_train = round(batch_size * len_olympus_train / len_train)
B_Z_train = batch_size - B_A_train - B_L_train - B_P_train - B_O_train


#batch sizes for validation
B_A_val = round(batch_size * len_akoya_val / len_val)
B_L_val = round(batch_size * len_leica_val / len_val)
B_P_val = round(batch_size * len_philips_val / len_val)
B_O_val = round(batch_size * len_olympus_val / len_val)
B_Z_val = batch_size - B_A_val - B_L_val - B_P_val - B_O_val


def make_loader(dataset, auto_batch_size):
    return DataLoader(dataset, batch_size=auto_batch_size, shuffle=True) #, num_workers=1, pin_memory=True, persistent_workers=True, prefetch_factor=4)

akoya_loader_train = make_loader(akoya_data_train, B_A_train)
leica_loader_train = make_loader(leica_data_train, B_L_train)
philips_loader_train = make_loader(philips_data_train, B_P_train)
olympus_loader_train = make_loader(olympus_data_train, B_O_train)
zeiss_loader_train = make_loader(zeiss_data_train, B_Z_train)



print("Train batches Akoya:", len(akoya_data_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_data_train), 'batch size:', B_L_train)
print("Train batches Philips:", len(philips_data_train), 'batch size:', B_P_train)
print("Train batches olympus:", len(olympus_data_train), 'batch size:', B_O_train)
print("Train batches Zeiss:", len(zeiss_data_train), 'batch size:', B_Z_train)
print('len train:', len_train)


akoya_loader_val = make_loader(akoya_data_val, B_A_val)
leica_loader_val = make_loader(leica_data_val, B_L_val)
philips_loader_val = make_loader(philips_data_val, B_P_val)
olympus_loader_val = make_loader(olympus_data_val, B_O_val)
zeiss_loader_val = make_loader(zeiss_data_val, B_Z_val)

print("Val batches Akoya:", len(akoya_data_val), 'batch size:', B_A_val)
print("Val batches Leica:", len(leica_data_val), 'batch size:', B_L_val)
print("Val batches Philips:", len(philips_data_val), 'batch size:', B_P_val)
print("Val batches olympus:", len(olympus_data_val), 'batch size:', B_O_val)
print("Val batches Zeiss:", len(zeiss_data_val), 'batch size:', B_Z_val)
print('len train:', len_val)




# Defining OT-based loss function
loss_geom = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)
Lambda = 0.1 # strength of OT (0.1 is the value of the article)

class NetworkHandler:
    '''
    A class to handle training, inference and prediction
    '''

    def __init__(self, precision = 'mixed', freeze_encoder = True, emb_mode = False, display = False):
        self.precision = precision
        self.freeze_encoder = freeze_encoder
        self.emb_mode = emb_mode
        self.display = display

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = Network(emb_mode=self.emb_mode)
        self.model = self.model.to(self.device, non_blocking=True)

        #printing architecture
        '''print("Bottleneck layers:")
        print(self.model.bottle_neck)
        print("Head layer:")
        print(self.model.head)
        if self.model.encoder is not None:
            print("\nEncoder architecture:")
            print(self.model.encoder)'''

        self.use_amp = precision == 'mixed' and self.device == 'cuda'
        self.grad_scaler = GradScaler(enabled=self.use_amp)

    
                  
    @torch.no_grad()
    def inference(self, custom_name, scanner, data_loader, visual=True):
        weights = f'/home/leolr-int/nfs/transformed_data/weights/{custom_name}/checkpoint.pth'
        checkpoint = torch.load(weights, weights_only=False, map_location=self.device)
        self.model.load_state_dict(checkpoint["model"])        
        
        self.model.eval()
        all_preds, all_labels, all_embeddings = [], [], []
        for batch in tqdm(data_loader, desc='Inference in progress...'):
            vectors= batch['embedding'].to(self.device)
            labels = batch['label'].to(self.device)
            
            with torch.autocast(device_type=self.device, dtype=torch.float16, enabled=self.use_amp):
                emb = self.model.bottle_neck(vectors)
                logits = self.model.head(emb)
                
            confidence = F.softmax(logits, dim=1)
            pred = torch.argmax(confidence, dim=1)
            
            all_preds.append(pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_embeddings.append(emb.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate (all_labels)
        all_embeddings = np.concatenate(all_embeddings)

        acc_score = balanced_accuracy_score(all_labels, all_preds)
        print(acc_score)

        CM = better_confusion_matrix(custom_name, all_labels, all_preds, scanner, acc_score)
        if visual:
            #random selection of 20% of the sample
            N = all_embeddings.shape[0]
            sample_size = int(0.2 * N)
            indices = torch.randperm(N)[:sample_size]
            sampled_embeddings = all_embeddings[indices]
            sampled_labels = all_labels[indices]
            dim_reduc_plot(embeddings=sampled_embeddings, y_true=sampled_labels, scanner=scanner, custom_name=custom_name, n_components=2)


    def training_OT(self, custom_name, num_epochs=20): 
        # we differentiate explicitly source and target scanner to apply the OT loss
        # training with validation

        training_stats = []
        min_loss_val, max_accuracy_val = float("inf"), -float('inf')
        
        #min_len_train = min(len(akoya_loader_train), len(leica_loader_train))
        #min_len_val = min(len(akoya_loader_val), len(leica_loader_val))
        
        trainable_params = list(filter(lambda p: p.requires_grad, self.model.parameters()))
        optimizer = torch.optim.AdamW(trainable_params, lr=10e-4, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

        ce_loss = nn.CrossEntropyLoss()  

        for epoch in range(1,num_epochs+1):
            metrics_train = {'running_loss': 0, 'predictions': [], 'labels': []}
            metrics_val   = {'running_loss': 0, 'predictions': [], 'labels': []} 

            count_train = 0
            count_val = 0
                            
            start = time.time()
            
            #1st part: train for one epoch
            self.model.train()

            #deactivate the encoder training if needed
            if self.freeze_encoder and not self.emb_mode:
                self.model.encoder.eval()

            #the target scanner is Akoya

            del_count = 0
            
            for batch_akoya, batch_leica, batch_philips, batch_olympus, batch_zeiss in tqdm(zip(akoya_loader_train, leica_loader_train, philips_loader_train, olympus_loader_train, zeiss_loader_train),
                                                    desc=f"Epoch {epoch} - Training Multi Scanner",
                                                    total = min(len(akoya_loader_train), len(leica_loader_train), len(philips_loader_train), len(olympus_loader_train), len(zeiss_loader_train))):
            
                del_count += 1
                count_train += len(batch_akoya) + len(batch_leica) + len(batch_philips) + len(batch_olympus) + len(batch_zeiss)
                #patches_akoya = (batch_akoya['embedding'] if self.emb_mode else batch_akoya['img']).to(self.device, non_blocking=True)
                #patches_leica = (batch_leica['embedding'] if self.emb_mode else batch_leica['img']).to(self.device, non_blocking=True)
        
                patch_akoya = batch_akoya['embedding'].to(self.device, non_blocking=True) 
                patch_leica = batch_leica['embedding'].to(self.device, non_blocking=True)   
                patch_philips = batch_philips['embedding'].to(self.device, non_blocking=True) 
                patch_olympus = batch_olympus['embedding'].to(self.device, non_blocking=True)
                patch_zeiss = batch_zeiss['embedding'].to(self.device, non_blocking=True)

                labels_akoya = batch_akoya['label'].to(self.device, non_blocking=True)
                labels_leica = batch_leica['label'].to(self.device, non_blocking=True)
                labels_philips = batch_philips['label'].to(self.device, non_blocking=True)
                labels_olympus = batch_olympus['label'].to(self.device, non_blocking=True)
                labels_zeiss = batch_zeiss['label'].to(self.device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
    
                with torch.autocast(device_type = self.device, dtype = torch.float16, enabled = self.use_amp):
                    
                    embedding_akoya = self.model.bottle_neck(patch_akoya) #.to(self.device, non_blocking=True)
                    embedding_leica = self.model.bottle_neck(patch_leica) #.to(self.device, non_blocking=True)
                    embedding_philips = self.model.bottle_neck(patch_philips)
                    embedding_olympus = self.model.bottle_neck(patch_olympus)
                    embedding_zeiss = self.model.bottle_neck(patch_zeiss)

                    logits_akoya = self.model(patch_akoya) #.to(self.device, non_blocking=True) 
                    logits_leica = self.model(patch_leica) #.to(self.device, non_blocking=True)
                    logits_philips = self.model(patch_philips)
                    logits_olympus = self.model(patch_olympus)
                    logits_zeiss = self.model(patch_zeiss)
                    
                    #OT_loss_train = loss_geom(embedding_akoya, embedding_leica)
                    #OT_loss_train = supervised_OT_loss(embedding_akoya, embedding_leica, labels_akoya, labels_leica)
                    #selective OT
                    #cost_fn_high = make_cost_fn(labels_akoya, labels_leica, p=p_penalty)
                    #loss_high = SamplesLoss(loss="sinkhorn", p=2, blur=0.05, scaling=0.95, backend="tensorized", cost=cost_fn_high)
                    #OT_loss_train = loss_high(embedding_akoya.detach(), embedding_leica)

                    loss_train = (ce_loss(logits_akoya, labels_akoya)
                                  + ce_loss(logits_leica, labels_leica)
                                  + ce_loss(logits_philips, labels_philips)
                                  + ce_loss(logits_olympus, labels_olympus)
                                  + ce_loss(logits_zeiss, labels_zeiss)
                                  + 0.1 * loss_geom(embedding_akoya, embedding_leica)
                                  + 0.1 * loss_geom(embedding_akoya, embedding_philips)
                                  + 0.1 * loss_geom(embedding_akoya, embedding_olympus)
                                  + 0.1 * loss_geom(embedding_akoya, embedding_zeiss))
                    
                self.grad_scaler.scale(loss_train).backward()
                self.grad_scaler.step(optimizer)
                self.grad_scaler.update()
                
                pred_akoya = torch.argmax(F.softmax(logits_akoya, dim=1), dim=1)
                pred_leica = torch.argmax(F.softmax(logits_leica, dim=1), dim=1)
                pred_philips = torch.argmax(F.softmax(logits_philips, dim=1), dim=1)
                pred_olympus = torch.argmax(F.softmax(logits_olympus, dim=1), dim=1)
                pred_zeiss = torch.argmax(F.softmax(logits_zeiss, dim=1), dim=1)
                
                #performance metrics
                
                metrics_train['running_loss'] += loss_train.detach().cpu().item()
                # we concatenate the predictions of source and target
                metrics_train['predictions'].extend(
                    np.concatenate([
                        pred_akoya.detach().cpu().numpy(),
                        pred_leica.detach().cpu().numpy(),
                        pred_philips.detach().cpu().numpy(),
                        pred_olympus.detach().cpu().numpy(),
                        pred_zeiss.detach().cpu().numpy()
                    ])
                )
                metrics_train['labels'].extend(
                    np.concatenate([
                        labels_akoya.detach().cpu().numpy(),
                        labels_leica.detach().cpu().numpy(),
                        labels_philips.detach().cpu().numpy(),
                        labels_olympus.detach().cpu().numpy(),
                        labels_zeiss.detach().cpu().numpy()
                    ])
                )

                if del_count % 50 == 0:  # Every 50 batches
                    torch.cuda.empty_cache()
                    
                # Delete intermediate tensors
                del patch_akoya, patch_leica, patch_philips, patch_olympus, patch_zeiss
                del labels_akoya, labels_leica, labels_philips, labels_olympus, labels_zeiss
                del logits_akoya, logits_leica, logits_philips, logits_olympus, logits_zeiss
                del pred_akoya, pred_leica, pred_philips, pred_olympus, pred_zeiss
                del loss_train

            epoch_loss_train = metrics_train['running_loss'] / count_train 
            epoch_balanced_accuracy_train = balanced_accuracy_score(metrics_train['labels'], metrics_train['predictions'])               
                
          
            # 2nd part: validation for one epoch 
            with torch.no_grad():
                self.model.eval()
                
                #we still work with the Train folder
                del_count = 0
                
                for batch_akoya, batch_leica, batch_philips, batch_olympus, batch_zeiss in tqdm(zip(akoya_loader_val, leica_loader_val, philips_loader_val, olympus_loader_val, zeiss_loader_val),
                                                    desc=f"Epoch {epoch} - Validation Multi Scanner",
                                                    total = min(len(akoya_loader_val), len(leica_loader_val), len(philips_loader_val), len(olympus_loader_val), len(zeiss_loader_val))):

                    del_count += 1
                    count_val += len(batch_akoya) + len(batch_leica) + len(batch_philips) + len(batch_olympus) + len(batch_zeiss)
                
                    patch_akoya = batch_akoya['embedding'].to(self.device, non_blocking=True) 
                    patch_leica = batch_leica['embedding'].to(self.device, non_blocking=True)   
                    patch_philips = batch_philips['embedding'].to(self.device, non_blocking=True) 
                    patch_olympus = batch_olympus['embedding'].to(self.device, non_blocking=True)
                    patch_zeiss = batch_zeiss['embedding'].to(self.device, non_blocking=True)

                    labels_akoya = batch_akoya['label'].to(self.device, non_blocking=True)
                    labels_leica = batch_leica['label'].to(self.device, non_blocking=True)
                    labels_philips = batch_philips['label'].to(self.device, non_blocking=True)
                    labels_olympus = batch_olympus['label'].to(self.device, non_blocking=True)
                    labels_zeiss = batch_zeiss['label'].to(self.device, non_blocking=True)

        
                    with torch.autocast(device_type = self.device, dtype = torch.float16, enabled = self.use_amp):
                        
                        embedding_akoya = self.model.bottle_neck(patch_akoya) #.to(self.device, non_blocking=True)
                        embedding_leica = self.model.bottle_neck(patch_leica) #.to(self.device, non_blocking=True)
                        embedding_philips = self.model.bottle_neck(patch_philips)
                        embedding_olympus = self.model.bottle_neck(patch_olympus)
                        embedding_zeiss = self.model.bottle_neck(patch_zeiss)

                        logits_akoya = self.model(patch_akoya) #.to(self.device, non_blocking=True) 
                        logits_leica = self.model(patch_leica) #.to(self.device, non_blocking=True)
                        logits_philips = self.model(patch_philips)
                        logits_olympus = self.model(patch_olympus)
                        logits_zeiss = self.model(patch_zeiss)
                        

                        loss_val = (ce_loss(logits_akoya, labels_akoya)
                                    + ce_loss(logits_leica, labels_leica)
                                    + ce_loss(logits_philips, labels_philips)
                                    + ce_loss(logits_olympus, labels_olympus)
                                    + ce_loss(logits_zeiss, labels_zeiss)
                                    + 0.1 * loss_geom(embedding_akoya.detach(), embedding_leica)
                                    + 0.1 * loss_geom(embedding_akoya.detach(), embedding_philips)
                                    + 0.1 * loss_geom(embedding_akoya.detach(), embedding_olympus)
                                    + 0.1 * loss_geom(embedding_akoya.detach(), embedding_zeiss))
                    
                    pred_akoya = torch.argmax(F.softmax(logits_akoya, dim=1), dim=1)
                    pred_leica = torch.argmax(F.softmax(logits_leica, dim=1), dim=1)
                    pred_philips = torch.argmax(F.softmax(logits_philips, dim=1), dim=1)
                    pred_olympus = torch.argmax(F.softmax(logits_olympus, dim=1), dim=1)
                    pred_zeiss = torch.argmax(F.softmax(logits_zeiss, dim=1), dim=1)
                    
                    #performance metrics
                
                    metrics_val['running_loss'] += loss_val.detach().cpu().item()
                    # we concatenate the predictions of source and target
                    metrics_val['predictions'].extend(
                        np.concatenate([
                            pred_akoya.detach().cpu().numpy(),
                            pred_leica.detach().cpu().numpy(),
                            pred_philips.detach().cpu().numpy(),
                            pred_olympus.detach().cpu().numpy(),
                            pred_zeiss.detach().cpu().numpy()
                        ])
                    )
                    metrics_val['labels'].extend(
                        np.concatenate([
                            labels_akoya.detach().cpu().numpy(),
                            labels_leica.detach().cpu().numpy(),
                            labels_philips.detach().cpu().numpy(),
                            labels_olympus.detach().cpu().numpy(),
                            labels_zeiss.detach().cpu().numpy()
                        ])
                    )

                    if del_count % 50 == 0:
                        torch.cuda.empty_cache()
                        
                    # Delete tensors
                    del patch_akoya, patch_leica, patch_philips, patch_olympus, patch_zeiss
                    del labels_akoya, labels_leica, labels_philips, labels_olympus, labels_zeiss
                    del logits_akoya, logits_leica, logits_philips, logits_olympus, logits_zeiss
                    del pred_akoya, pred_leica, pred_philips, pred_olympus, pred_zeiss
                    del loss_val
        
        
                epoch_loss_val = metrics_val['running_loss'] / count_val
                
                epoch_balanced_accuracy_val = balanced_accuracy_score(metrics_val['labels'], metrics_val['predictions'])
                
                scheduler.step(epoch_loss_val) 
                
                cm = confusion_matrix(metrics_val["labels"], metrics_val["predictions"], labels=[0, 1, 2, 3, 4], normalize='true')
    
                end = time.time()
    
                dic = {'epoch_loss_train': epoch_loss_train, 
                    'epoch_balanced_accuracy_train': epoch_balanced_accuracy_train, 
                    'epoch_loss_val': epoch_loss_val, 
                    'epoch_balanced_accuracy_val': epoch_balanced_accuracy_val, 
                    'time': end - start,
                    'cm':cm}
                
                training_stats.append(dic)

                min_loss_val, max_accuracy_val = end_epoch(
                                                    save_dir,
                                                    custom_name,
                                                    self.model,
                                                    optimizer,
                                                    scheduler,
                                                    epoch,
                                                    epoch_loss_train,
                                                    epoch_balanced_accuracy_train,
                                                    epoch_loss_val,
                                                    epoch_balanced_accuracy_val,
                                                    min_loss_val,
                                                    max_accuracy_val)
                
                train_plot(pd.DataFrame(training_stats), cm, custom_name=custom_name, OT=False)
                torch.cuda.empty_cache()

        return epoch_loss_val, epoch_balanced_accuracy_val


        
if __name__ == '__main__':
    handler = NetworkHandler(emb_mode=True)
    save_dir = '/home/leolr-int/nfs/transformed_data/weights'
    custom_name = 'OT_multi_no_detach'
    num_epochs = 20
    handler.training_OT(custom_name, num_epochs=num_epochs)

Train batches Akoya: 1984027 batch size: 242
Train batches Leica: 562417 batch size: 69
Train batches Philips: 639820 batch size: 78
Train batches olympus: 522381 batch size: 64
Train batches Zeiss: 489769 batch size: 59
len train: 4198414
Val batches Akoya: 493493 batch size: 224
Val batches Leica: 156995 batch size: 71
Val batches Philips: 179706 batch size: 82
Val batches olympus: 141792 batch size: 64
Val batches Zeiss: 156338 batch size: 71
len train: 1128324


Epoch 1 - Training Multi Scanner:   0%|                                            | 1/8151 [00:09<22:05:03,  9.76s/it]

In [ ]:
#DONT FORGET INFERENCE!!!!!